# Part1

In [1]:
import sentence_transformers
import transformers
import huggingface_hub
import langchain_huggingface

print(sentence_transformers.__version__)
print(transformers.__version__)
print(huggingface_hub.__version__)


c:\conda\envs\venv\lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange


3.0.1
4.46.1
0.33.4


In [2]:
import os
from langchain_groq import ChatGroq
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.prompts import ChatPromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.embeddings import HuggingFaceEmbeddings

from dotenv import load_dotenv
load_dotenv()

# LLM
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0
)

# Embeddings
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# PDF path (YOU will change this)
PDF_PATH = "C:/Users/kumar/OneDrive/Desktop/TRY-3/Tasks-Submission/Assignment34/4 - Harry Potter and the Goblet of Fire.pdf"


C:\Windows\Temp\ipykernel_16656\2906872471.py:21: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


# Task1

In [3]:
# Load PDF document
loader = PyPDFLoader(PDF_PATH)
documents = loader.load()

# Combine all pages into one text
full_text = "\n".join([doc.page_content for doc in documents])

# Print stats
print("Total Characters:", len(full_text))
print("\nSample Preview:\n")
print(full_text[:500])


Total Characters: 1094523

Sample Preview:



Harry Potter
and the Goblet Of Fire
 
 
by
J. K. Rowling
Illustrations by Mary Grandpré
 
 
 
 
Arthur A. Levine Books
An Imprint of Scholastic Press
To Peter Rowling,
In Memory of Mr. Ridley
And to Susan Sladden,
Who helped Harry
Out of his cupboard
Text copyright © 2000 by J.K. Rowling
Illustrations by Mary GrandPre copyright © 2000 Warner
Bros.
All rights reserved. Published by Scholastic Press, a division
of Scholastic Inc.,
Publishers since 1920.
SCHOLASTIC, SCHOLASTIC PRESS, and the LANT


# Task2

In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,   # safe for Groq
    chunk_overlap=150
)

chunks = splitter.split_text(full_text)

print(f"Total chunks: {len(chunks)}")


Total chunks: 1285


In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_template("""
You are a professional document summarizer.
Summarize the following text clearly and concisely.

Text:
{text}
""")

summary_chain = prompt | llm | StrOutputParser()


In [14]:
import numpy as np

# pick 10 evenly spaced chunks
indices = np.linspace(0, len(chunks) - 1, 10, dtype=int)
selected_chunks = [chunks[i] for i in indices]


In [15]:
combined_text = "\n\n".join(selected_chunks)

summary = summary_chain.invoke({
    "text": combined_text
})

print(summary)


Here is a clear and concise summary of the provided text:

The story begins with Harry Potter, Ron Weasley, and Hermione Granger arriving at the Quidditch World Cup, where they witness a dramatic event involving dragons and a mysterious ship rising out of the lake. 

After the event, they return to Hogwarts School of Witchcraft and Wizardry, where they are greeted by their friends and professors. However, the atmosphere is tense due to the recent disappearance of a student, and Harry, Ron, and Hermione are tasked with helping to find him.

Meanwhile, Harry receives a mysterious package from Dobby, a house-elf, containing socks that Dobby has knitted himself. Harry also visits the library to research underwater survival, as he is concerned about a possible connection between the missing student and the lake.

The story then shifts to a flashback of Lord Voldemort's past, where he is recounting his failed attempt to conquer death. The chapter ends with Harry saying goodbye to his friends

# Task3

In [17]:
from langchain_core.prompts import PromptTemplate
short_prompt = PromptTemplate(
    input_variables=["text"],
    template="""
Summarize the following text in 5–6 concise lines.

Text:
{text}
"""
)

short_summary = llm.invoke(short_prompt.format(text=summary))
print(short_summary.content)


Here's a 5-6 line summary of the text:

Harry Potter, Ron Weasley, and Hermione Granger attend the Quidditch World Cup, witnessing a dramatic event involving dragons and a mysterious ship. They return to Hogwarts, where they help find a missing student. Harry receives a mysterious package from Dobby and researches underwater survival. A flashback reveals Lord Voldemort's failed attempt to conquer death. The chapter ends with Harry saying goodbye to friends and returning home.


# Part2

# Task4

What is Stuff Chain?
→ Passes entire document at once to the LLM.

When suitable?
→ Short documents that fit context window.

Limitations?
→ Fails for long documents, token overflow.

# Task5

In [21]:
stuff_prompt = ChatPromptTemplate.from_template("""
Summarize the following document.

{text}
""")

stuff_chain = stuff_prompt | llm | StrOutputParser()

stuff_summary = stuff_chain.invoke({"text": summary})
print(stuff_summary)


The provided text is not a document but a summary of a story. However, I can summarize the summary for you:

The story begins with Harry Potter, Ron, and Hermione attending the Quidditch World Cup, where they witness a mysterious event involving a ship and dragons. Upon returning to Hogwarts, they help find a missing student and Harry receives a mysterious package from Dobby. The story also includes a flashback of Voldemort's past and Harry's preparations to face challenges ahead.


# Task6

Prompt summary → More controllable

Stuff chain → Simpler, but limited by context

# Part3

# Task7

Needed for large documents

Map → summarize chunks

Reduce → combine summaries

# Task8

In [23]:
map_prompt = ChatPromptTemplate.from_template("""
Summarize this chunk briefly.

{text}
""")

map_chain = map_prompt | llm | StrOutputParser()

map_summaries = [
    map_chain.invoke({"text": summary})
    for chunk in summary
]


KeyboardInterrupt: 

In [ ]:
reduce_prompt = ChatPromptTemplate.from_template("""
Combine the following summaries into a coherent final summary.

{text}
""")

reduce_chain = reduce_prompt | llm | StrOutputParser()

final_summary = reduce_chain.invoke({
    "text": "\n\n".join(map_summaries)
})

print(final_summary)


# Task9

In [ ]:
map_outputs = map_reduce_chain.combine_documents_chain.llm_chain.prompt
print("Map step combines chunk summaries internally.")


# Part4

# Task10

Refine improves summary iteratively

Better coherence than map-reduce

Slower but higher quality

# Task11

In [ ]:
refine_chain = load_summarize_chain(
    llm,
    chain_type="refine"
)

refine_summary = refine_chain.invoke(split_docs)
print(refine_summary["output_text"])


# Task12

map-reduce and refine has good and best quality whereas prompt and stuff has medium queality, refine has highest coherence and good for long docs.

# Part5

# Task13

In [ ]:
def summarize_document(text, method="map_reduce"):
    docs_local = [type(docs[0])(page_content=text, metadata={})]

    if method == "prompt":
        return llm.invoke(prompt.format(text=text)).content

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200
    )
    split_docs = splitter.split_documents(docs_local)

    chain = load_summarize_chain(llm, chain_type=method)
    return chain.invoke(split_docs)["output_text"]


# Task14

In [ ]:
"""
1. Best for very long docs: Map-Reduce
2. Best quality: Refine
3. Trade-off:
   - Speed vs coherence
4. Real-world use:
   - Prompt: emails
   - Stuff: blogs
   - Map-Reduce: research papers
   - Refine: legal / policy documents
"""
